# 머신러닝 기반 텍스트분류
1. 데이터 준비 : 로딩, 입력데이터, 출력데이터, 학습/테스트 데이터 분리, 특징추출
2. 학습 - 평가
3. 배포 준비

### 1. 데이터 준비

In [3]:
import pandas as pd
datafile = "./data/Korean_movie_reviews_2016.csv"
data_df = pd.read_csv(datafile)
data_df.head()

,review,label
0,부산 행 때문 너무 기대하고 봤,0
1,한국 좀비 영화 어색하지 않게 만들어졌 놀랍,1
2,조금 전 보고 왔 지루하다 언제 끝나 이 생각 드,0
3,평 밥 끼 먹자 돈 니 내고 미친 놈 정신사 좀 알 싶어 그래 밥 먹다 먹던 숟가락...,1
4,점수 대가 과 이 엑소 팬 어중간 점수 줄리 없겠 클레멘타인 이후 최고 평점 조작 ...,0


In [4]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165384 entries, 0 to 165383
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   review  165384 non-null  object
 1   label   165384 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


In [5]:
# 입력 데이터, 정답 데이터 추출
review_list = list(data_df.review) # 입력 데이터
label_list = list(data_df.label) # 출력 데이터
len(review_list), len(label_list)

(165384, 165384)

In [6]:
# 학습 데이터, 테스트 데이터 분리
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(review_list, label_list, test_size=0.1)
len(train_X), len(test_X), len(train_y), len(test_y)

(148845, 16539, 148845, 16539)

In [7]:
# 한국어 토크나이저 정의
from konlpy.tag import Okt
def korean_tokenizer(text):
    my_tags = ['Nouns', 'Adjective', 'Verb']
    my_stopwords = []
    tokenizer = Okt().pos
    return [word for word, tag in tokenizer(text) if tag in my_tags and word not in my_stopwords]

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000)
vectorizer.fit(train_X)

TfidfVectorizer(max_features=1000)

In [38]:
len(vectorizer.get_feature_names_out()), vectorizer.get_feature_names_out()[:10]

(1000,
 array(['가고', '가는', '가면', '가볍', '가서', '가슴', '가장', '가족', '가지', '가치'],
       dtype=object))

In [42]:
# 학습 데이터 특징 추출
train_X_fv = vectorizer.transform(train_X)

In [43]:
# 테스트 데이터 특징 추출
test_X_fv = vectorizer.transform(test_X)

In [44]:
print(train_X_fv)

  (np.int32(0), np.int32(95))	0.7624408683844051
  (np.int32(0), np.int32(969))	0.6470578971137236
  (np.int32(1), np.int32(1))	0.404061447934304
  (np.int32(1), np.int32(155))	0.22854136019547563
  (np.int32(1), np.int32(303))	0.34463146456990773
  (np.int32(1), np.int32(480))	0.2987779713559235
  (np.int32(1), np.int32(589))	0.47148501872075976
  (np.int32(1), np.int32(775))	0.3947672933822989
  (np.int32(1), np.int32(781))	0.36868949980691307
  (np.int32(1), np.int32(868))	0.2497854766869
  (np.int32(2), np.int32(482))	0.6737207693579816
  (np.int32(2), np.int32(636))	0.22576422272395308
  (np.int32(2), np.int32(935))	0.7036553422475657
  (np.int32(3), np.int32(20))	0.4003863277391233
  (np.int32(3), np.int32(602))	0.2893822049539147
  (np.int32(3), np.int32(627))	0.30528653005685136
  (np.int32(3), np.int32(630))	0.35450756753681045
  (np.int32(3), np.int32(761))	0.4618900984122546
  (np.int32(3), np.int32(920))	0.4333071442403258
  (np.int32(3), np.int32(958))	0.36874883441115053


In [45]:
# 정답 데이터를 ndarry 변환 np.array(sequence)
import numpy as np
train_y = np.array(train_y)
test_y = np.array(test_y)
train_y[:10]

array([0, 1, 0, 0, 1, 0, 1, 0, 1, 0])

# 2. 머신러닝 - 모델 학습
1. 의사결정트리, Decision Tree (DT)
2. 랜덤포레트스, Random Forest (RF)

In [14]:
# 모델별 정확도를 dataframe으로 저장하여 비교
import pandas as pd
score_df = pd.DataFrame(columns=['train', 'test'])
score_df

,train,test


In [15]:
def get_scores(model, train_X, train_y, test_X, test_y):
    train_score = model.score(train_X, train_y) * 100
    test_score = model.score(test_X, test_y) * 100
    return train_score, test_score

## 1. Decision Tree

In [48]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()
dtc.fit(train_X_fv, train_y)

DecisionTreeClassifier()

In [49]:
train_score, test_score = get_scores(dtc, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)

95.83056199402063 79.67833605417498


In [50]:
score_df.loc['DecisionTree'] = [train_score, test_score] # 행을 지정하고 싶으면 loc를 씀
score_df

,train,test
DecisionTree,95.830562,79.678336
RandomForest,95.827203,83.384727
NaiveBayes,77.422151,77.296088
LogisticRegression,77.716416,77.568172
SVM,77.680137,77.628635


## 2. Random Forest

In [46]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs=-1)
rf.fit(train_X_fv, train_y)

RandomForestClassifier(n_jobs=-1)

In [47]:
train_score, test_score = get_scores(rf, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['RandomForest'] = [train_score, test_score] # 행을 지정하고 싶으면 loc를 씀
score_df

95.8272027948537 83.38472700888808


,train,test
DecisionTree,87.751016,73.656207
RandomForest,95.827203,83.384727
NaiveBayes,77.422151,77.296088
LogisticRegression,77.716416,77.568172
SVM,77.680137,77.628635


## 3. 나이브 베이즈 분류


In [21]:
from sklearn.naive_bayes import MultinomialNB
mnb = MultinomialNB()
mnb.fit(train_X_fv, train_y)

MultinomialNB()

In [22]:
train_score, test_score = get_scores(mnb, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['NaiveBayes'] = [train_score, test_score] # 행을 지정하고 싶으면 loc를 씀
score_df

77.42215055930666 77.29608803434307


,train,test
DecisionTree,87.751016,73.656207
RandomForest,87.749672,76.316585
NaiveBayes,77.422151,77.296088


## 4. 로지스틱 회귀 분석, logistic regression

In [23]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(solver='liblinear')
lr.fit(train_X_fv, train_y)

LogisticRegression(solver='liblinear')

In [24]:
train_score, test_score = get_scores(lr, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['LogisticRegression'] = [train_score, test_score] # 행을 지정하고 싶으면 loc를 씀
score_df

77.71641640632873 77.56817219904468


,train,test
DecisionTree,87.751016,73.656207
RandomForest,87.749672,76.316585
NaiveBayes,77.422151,77.296088
LogisticRegression,77.716416,77.568172


## 5. 서포트 벡터 머신, SVM

In [25]:
from sklearn.svm import LinearSVC
svm = LinearSVC()
svm.fit(train_X_fv, train_y)

LinearSVC()

In [36]:
train_score, test_score = get_scores(svm, train_X_fv, train_y, test_X_fv, test_y)
print(train_score, test_score)
score_df.loc['SVM'] = [train_score, test_score] # 행을 지정하고 싶으면 loc를 씀
score_df

77.68013705532601 77.62863534675614


,train,test
DecisionTree,87.751016,73.656207
RandomForest,87.749672,76.316585
NaiveBayes,77.422151,77.296088
LogisticRegression,77.716416,77.568172
SVM,77.680137,77.628635


In [35]:
score_df.sort_values(by='test', ascending=False) # 여러개 기준으로 분류하고 싶으면 리스트로 하면됨

,train,test
SVM,77.680137,77.628635
LogisticRegression,77.716416,77.568172
NaiveBayes,77.422151,77.296088
RandomForest,87.749672,76.316585
DecisionTree,87.751016,73.656207


## 6. 배포 준비
- 기능 구현
- 모델 저장

In [ ]:
# review = '영화가 너무 재미있다'
# review = '이게 영화냐? 나도 만들겠다'
# review = '영화 보는 내내 하품만 했다'
review = '재미가 없다'

def analyze_sentiment(review):
    # 전처리 및 특징 벡터 추출
    
    review_fv = vectorizer.transform([review])
    # print(review_fv)

    result = rf.predict(review_fv)
    # print(result)

    show = '긍정' if result[0] >= 0.5 else '부정'
    return show

show = analyze_sentiment(review)
print(f'{review} -> {show}')

재미가 없다 -> 부정


In [33]:
reviews = ['영화가 너무 재미있다',
           '이게 영화냐? 나도 만들겠다',
           '영화 보는 내내 하품만 했다',
           '재미가 없다',
           '개꿀잼',
           '핵노잼']

for review in reviews:
    print(f'{review} -> {analyze_sentiment(review)}')

영화가 너무 재미있다 -> 긍정
이게 영화냐? 나도 만들겠다 -> 긍정
영화 보는 내내 하품만 했다 -> 부정
재미가 없다 -> 부정
개꿀잼 -> 긍정
핵노잼 -> 긍정


In [34]:
import joblib

vectorizer_file = './model/sa_movie_vectorizer.pkl'
joblib.dump(vectorizer, vectorizer_file)

model_file = './model/sa_movie_model.pkl'
joblib.dump(rf, model_file)

['./model/sa_movie_model.pkl']